# Problems with RNNs: vanishing and exploding gradients through time
1. The scalar picture: a product of (tanh slope x w_h) over many steps.
2. The same W_h, many times: a linear SimpleRNN on IMDB reviews with W_h = s x (orthogonal matrix).
3. A SimpleRNN trained on IMDB: how much each word of a review moves the loss, by distance from the end.
4. Long-term dependency: the same reviews followed by a gap of blank steps; accuracy as the gap grows.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()   # same numbers on every run
DATA = Path("data")
DATA.mkdir(exist_ok=True)

## 1. The scalar picture
One hidden unit: dh_t/dh_(t-1) = tanh'(.) * w_h. Even with the largest possible tanh slope (1),
w_h = 0.9 shrinks the gradient of a word 100 steps back by 0.9**100.

In [ ]:
for w_h in (0.5, 0.9, 1.0, 1.1):
    print(f"w_h = {w_h}:  10 steps {w_h ** 10:.3g}   100 steps {w_h ** 100:.3g}")
print("tanh slope 0.8 and w_h 0.9 together, 100 steps:", f"{(0.8 * 0.9) ** 100:.3g}")

## Data: IMDB reviews, last 200 words of reviews that have at least 200

In [ ]:
NUM_WORDS, T = 10000, 200
(x_tr, y_tr), (x_te, y_te) = keras.datasets.imdb.load_data(num_words=NUM_WORDS)
long_te = [i for i, r in enumerate(x_te) if len(r) >= T][:500]
X_long = keras.utils.pad_sequences([x_te[i] for i in long_te], maxlen=T)    # keeps the last 200 words
Y_long = y_te[long_te].astype("float32")
print("test reviews with >= 200 words used:", len(long_te))


def input_gradients(model, X, Y):
    """Mean size (norm) of dL/dx_k, the gradient of the loss with respect to the word vector at step k."""
    emb, rest = model.layers[0], model.layers[1:]
    with tf.GradientTape() as tape:
        e = emb(X)
        tape.watch(e)
        out = e
        for layer in rest:
            out = layer(out)
        loss = keras.losses.binary_crossentropy(Y[:, None], out)
        loss = tf.reduce_sum(loss)
    g = tape.gradient(loss, e).numpy()                        # (reviews, T, embedding size)
    return np.linalg.norm(g, axis=2).mean(axis=0)              # one number per time step

## 2. The same W_h, many times
Linear activation (Goodfellow eq. 10.36), untrained, W_h = s * orthogonal matrix: every step multiplies the
gradient by W_h, so it should change by a factor s per step.

In [ ]:
rows = []
for s in (0.9, 1.0, 1.1):
    keras.utils.set_random_seed(1)
    m = keras.Sequential([keras.Input(shape=(T,)), keras.layers.Embedding(NUM_WORDS, 32),
                          keras.layers.SimpleRNN(32, activation="linear"), keras.layers.Dense(1, activation="sigmoid")])
    Q = np.linalg.qr(np.random.default_rng(0).normal(size=(32, 32)))[0]      # orthogonal: all |eigenvalues| = 1
    k, _, b = m.layers[1].get_weights()
    m.layers[1].set_weights([k, s * Q, b])
    g = input_gradients(m, X_long[:100], Y_long[:100])
    rows += [dict(scale=s, distance=T - 1 - t, grad=g[t]) for t in range(T)]
    print(f"s = {s}: gradient at distance 0 {g[-1]:.3g}, 50 {g[-51]:.3g}, 199 {g[0]:.3g};"
          f" ratio 50 steps {g[-51] / g[-1]:.3g} vs s**50 = {s ** 50:.3g}")
pd.DataFrame(rows).to_csv(DATA / "linear_scaled_wh.csv", index=False)

## 3. A SimpleRNN trained on IMDB (tanh)
10,000 training reviews, last 200 words, 3 epochs. Then the gradient of the loss with respect to each word.

In [ ]:
X_tr = keras.utils.pad_sequences(x_tr[:10000], maxlen=T)
Y_tr = y_tr[:10000]
X_te = keras.utils.pad_sequences(x_te[:5000], maxlen=T)
Y_te = y_te[:5000]

keras.utils.set_random_seed(42)
rnn = keras.Sequential([keras.Input(shape=(T,)), keras.layers.Embedding(NUM_WORDS, 32),
                        keras.layers.SimpleRNN(32), keras.layers.Dense(1, activation="sigmoid")])
rnn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
rnn.fit(X_tr, Y_tr, epochs=3, batch_size=64, verbose=2)
print("test accuracy:", round(rnn.evaluate(X_te, Y_te, verbose=0)[1], 3))
g = input_gradients(rnn, X_long, Y_long)
dist = np.arange(T)[::-1]                        # distance from the last word
pd.DataFrame(dict(distance=dist, grad=g)).to_csv(DATA / "trained_tanh.csv", index=False)
for d in (0, 5, 10, 20, 50, 100, 199):
    print(f"distance {d:3d}: mean gradient size {g[T - 1 - d]:.3g}  ({g[T - 1 - d] / g[-1]:.2g} x the last word)")

## 4. Long-term dependency: a gap after the review
Each review is cut to its last 50 words; then G blank steps (token 0) follow before the prediction.
Only G changes. The RNN must carry the review's sentiment across the gap.

In [ ]:
L_REV, results = 50, []
Xr_tr = keras.utils.pad_sequences(x_tr[:10000], maxlen=L_REV)
Xr_te = keras.utils.pad_sequences(x_te[:5000], maxlen=L_REV)
for gap in (0, 10, 20, 40, 80):
    for seed in (1, 2):
        keras.utils.set_random_seed(seed)
        add = lambda X: np.hstack([X, np.zeros((len(X), gap), dtype=X.dtype)])
        m = keras.Sequential([keras.Input(shape=(L_REV + gap,)), keras.layers.Embedding(NUM_WORDS, 32),
                              keras.layers.SimpleRNN(32), keras.layers.Dense(1, activation="sigmoid")])
        m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
        h = m.fit(add(Xr_tr), Y_tr, epochs=5, batch_size=64, verbose=0, validation_data=(add(Xr_te), Y_te))
        acc = h.history["val_accuracy"][-1]
        results.append(dict(gap=gap, seed=seed, test_accuracy=acc))
        print(f"gap {gap:3d} seed {seed}: test accuracy after 5 epochs {acc:.3f}")
res = pd.DataFrame(results)
res.to_csv(DATA / "gap_accuracy.csv", index=False)
print(res.groupby("gap").test_accuracy.mean().round(3))